[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgentOpt/Trace-Bench/blob/main/notebooks/07_multiobjective_bbeh.ipynb)

# Trace-Bench — Multi-Objective BBEH Boolean Expressions

This notebook benchmarks three trainer algorithms on the **BBEH boolean_expressions**
multi-objective task, comparing **weighted** vs **Pareto** selection modes across
two LLM backends.

**Task**: optimize Python code (PAL strategy) that evaluates boolean expressions.
The trainable parameter is a code template; the optimizer proposes code improvements.

| Objective | Description | Direction |
|-----------|-------------|-----------|
| `accuracy` | 1.0 correct, 0.0 wrong | maximize |
| `execution_time_s` | CPU seconds via `time.process_time()` | minimize |

**Trainers**: BasicSearchAlgorithm, BeamsearchAlgorithm, PrioritySearch — all with `batch_size=1`.

**Models**: `grok-4.1-fast` and `deepseek` (v3.2 or chat).
Supports both **OpenRouter** (`OPENROUTER_API_KEY`) and **direct provider keys**
(`XAI_API_KEY` + `DEEPSEEK_API_KEY`) via LiteLLM. Falls back to stub/DummyLLM
if no keys are found. Place keys in a `.env` file or export them in your shell.

**PAL strategy**: the optimizer proposes Python code that sets `result` to `'True'`
or `'False'`. Execution is timed with `time.process_time()` (CPU time, not wall-clock)
as specified by the benchmark spec.

## Expected Outputs

- Up to 12 completed jobs (3 trainers × 2 modes × 2 models) in two separate runs.
- Results table comparing `score_initial`, `score_best`, `time_seconds` across all runs.
- Score progression plots (accuracy + execution_time_s) per trainer/mode.
- Scatter plot of accuracy vs execution_time_s, faceted by model.
- Side-by-side model comparison bar chart.

In [ ]:
# Setup: persistent output dir, API key detection, model config
from datetime import date
from pathlib import Path
import os, sys

# --- Load .env (local dev) ---------------------------------------------------
try:
    from dotenv import load_dotenv
    # Walk up to find .env (repo root or parent)
    _here = Path.cwd()
    for _candidate in [_here, _here.parent, _here.parent.parent]:
        _envfile = _candidate / ".env"
        if _envfile.is_file():
            load_dotenv(_envfile, override=False)
            print(f"Loaded .env from {_envfile}")
            break
except ImportError:
    pass  # dotenv not installed — rely on shell environment

# --- Detect environment: Colab vs local --------------------------------------
ON_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except Exception:
    pass

if ON_COLAB:
    REPO_ROOT = Path("/content/Trace-Bench")
    OPENTRACE_ROOT = Path("/content/OpenTrace")
else:
    REPO_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
    if not (REPO_ROOT / "trace_bench").is_dir():
        REPO_ROOT = Path.cwd()
    OPENTRACE_ROOT = REPO_ROOT.parent / "OpenTrace"


def bench_dir(project="bench", sub="trace_bench_bbeh"):
    if ON_COLAB:
        drive_root = Path("/content/drive/MyDrive")
        root = drive_root if drive_root.is_dir() else Path("/content/bench")
    else:
        root = REPO_ROOT / "runs"
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)


RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)
print(f"Repo root: {REPO_ROOT}")
print(f"OpenTrace: {OPENTRACE_ROOT}")

# --- Auto-detect API provider ------------------------------------------------
# Priority: OpenRouter > Colab secrets > Direct provider keys > stub
#
# 1. OpenRouter  — models prefixed "openrouter/..."
# 2. Direct      — XAI_API_KEY + DEEPSEEK_API_KEY -> native LiteLLM model IDs
# 3. Stub        — DummyLLM fallback

PROVIDER = None
MODELS = []

# Try OpenRouter first (Xavier's default setup)
_or_key = os.environ.get("OPENROUTER_API_KEY", "")
if not _or_key:
    try:
        from google.colab import userdata
        _or_key = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass

if _or_key:
    PROVIDER = "openrouter"
    os.environ["OPENROUTER_API_KEY"] = _or_key
    os.environ["OPENAI_API_KEY"] = _or_key
    os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"
    os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODELS = [
        os.environ.get("TRACE_LITELLM_MODEL", "openrouter/x-ai/grok-4.1-fast"),
        os.environ.get("TRACE_LITELLM_MODEL_2", "openrouter/deepseek/deepseek-v3.2"),
    ]

# Try direct provider keys (XAI + DeepSeek via native LiteLLM routing)
if not PROVIDER:
    # Handle common XIA_API_KEY typo
    if not os.environ.get("XAI_API_KEY") and os.environ.get("XIA_API_KEY"):
        os.environ["XAI_API_KEY"] = os.environ["XIA_API_KEY"]

    _xai_key = os.environ.get("XAI_API_KEY", "")
    _ds_key = os.environ.get("DEEPSEEK_API_KEY", "")

    if _xai_key or _ds_key:
        PROVIDER = "direct"
        MODELS = []
        if _xai_key:
            MODELS.append("xai/grok-4-1-fast-non-reasoning")
        if _ds_key:
            MODELS.append("deepseek/deepseek-chat")

# Fallback to stub
if not PROVIDER or not MODELS:
    PROVIDER = "stub"
    MODELS = []

# Set mode and backend
if PROVIDER in ("openrouter", "direct"):
    os.environ["TRACE_DEFAULT_LLM_BACKEND"] = "LiteLLM"
    MODE = "real"
    print(f"Provider: {PROVIDER} — running in REAL mode")
    print(f"Models: {MODELS}")
else:
    MODE = "stub"
    print("WARNING: No API keys found. Falling back to STUB mode.")
    print("         Stub results use DummyLLM — not real optimization.")

os.environ["TB_MODE"] = MODE
print(f"\nMode: {MODE}")

In [ ]:
# Clone repos + install deps (Colab only — skipped locally)
if ON_COLAB:
    !git clone --depth 1 https://github.com/AgentOpt/Trace-Bench.git /content/Trace-Bench 2>/dev/null || (cd /content/Trace-Bench && git pull --ff-only)
    !git clone --depth 1 --branch experimental https://github.com/AgentOpt/OpenTrace.git /content/OpenTrace 2>/dev/null || (cd /content/OpenTrace && git pull)

    # Add repos to sys.path so trace_bench and opto are importable
    sys.path.insert(0, str(REPO_ROOT))
    sys.path.insert(0, str(OPENTRACE_ROOT))

    %cd /content/Trace-Bench
    !python -m pip install -q pyyaml pytest numpy matplotlib pandas litellm==1.75.0

    # Verify task module is available after clone
    _task_mod = REPO_ROOT / "trace_bench" / "examples" / "multiobjective_bbeh.py"
    assert _task_mod.exists(), f"ERROR: {_task_mod} not found — ensure main branch is up to date."
    print(f"Task module verified: {_task_mod}")
else:
    os.chdir(str(REPO_ROOT))
    print(f"Working directory: {Path.cwd()}")
    print("Skipping clone/install — running locally.")

## 1. Load Task Bundle

Verify the BBEH multi-objective task loads correctly and inspect the objective config.

In [ ]:
import sys

sys.path.insert(0, str(OPENTRACE_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from trace_bench.registry import load_task_bundle

bundle = load_task_bundle(
    "internal:multiobjective_bbeh",
    "benchmarks/LLM4AD/benchmark_tasks",
    eval_kwargs={"objective_mode": "weighted"},
)

print("Bundle keys:", sorted(bundle.keys()))
print(f"Param type: {type(bundle['param']).__name__}")
print(f"Param value (initial code):\n{bundle['param'].data}")
print(f"\nGuide type: {type(bundle['guide']).__name__}")
print(f"Metadata: {bundle['metadata']}")
print(f"Train examples: {len(bundle['train_dataset']['inputs'])}")

oc = bundle["objective_config"]
print(f"\nObjectiveConfig:")
print(f"  mode     = {oc.mode}")
print(f"  weights  = {dict(oc.weights)}")
print(f"  minimize = {set(oc.minimize)}")

# Quick sanity: evaluate initial code on first example
guide = bundle["guide"]
q = bundle["train_dataset"]["inputs"][0]
ref = bundle["train_dataset"]["infos"][0]
code = str(bundle["param"].data)
sd = guide.get_score_dict(query=q, response=code, reference=ref)
print(f"\nExample question: {q}")
print(f"Expected: {ref}")
print(f"Score dict: {sd}")
del bundle

## 2. Run Experiments

3 trainers × 2 modes (weighted / pareto) × 2 models = **12 experiments**.

Each model is run as a separate `trace_bench run` invocation with the
corresponding `TRACE_LITELLM_MODEL` environment variable.

Training params are conservative (`num_epochs=2`, `batch_size=1`) since
each step involves LLM API calls.

In [ ]:
# Write the YAML config (shared by both model runs)
import pathlib

yaml_content = """\
mode: {mode}
seeds: [42]
max_workers: 6  # 6 parallel jobs — bottleneck is API latency, not CPU
resume: auto
job_timeout: 1200

tasks:
  - id: "internal:multiobjective_bbeh"
    eval_kwargs:
      objective_mode: "weighted"

  - id: "internal:multiobjective_bbeh"
    eval_kwargs:
      objective_mode: "pareto"

trainers:
  - id: BasicSearchAlgorithm
    params_variants:
      - num_proposals: 4
        num_epochs: 2
        batch_size: 1

  - id: BeamsearchAlgorithm
    params_variants:
      - beam_width: 2
        num_proposals: 4
        max_depth: 2
        batch_size: 1

  - id: PrioritySearch
    params_variants:
      - num_candidates: 4
        num_proposals: 2
        batch_size: 1
""".format(mode=MODE)

config_path = pathlib.Path(RUNS_DIR) / "bbeh.yaml"
config_path.write_text(yaml_content)
print(f"Config written to {config_path}")
print(f"Mode: {MODE}")
print(yaml_content)

In [ ]:
# Run Model 1 (first model in MODELS list)
import subprocess, os, sys

if not MODELS:
    print("No models configured — skipping. Running in stub mode.")
    # Run stub instead
    MODEL_1 = "stub"
    MODEL_1_TAG = "stub"
else:
    MODEL_1 = MODELS[0]
    MODEL_1_TAG = MODEL_1.split("/")[-1]  # e.g. "grok-4.1-fast"

print(f"=== Running BBEH with {MODEL_1} (provider: {PROVIDER}) ===")

env = dict(os.environ)
env["PYTHONPATH"] = f"{OPENTRACE_ROOT}:{env.get('PYTHONPATH', '')}"
if MODEL_1 != "stub":
    env["TRACE_LITELLM_MODEL"] = MODEL_1

result = subprocess.run(
    [
        sys.executable, "-m", "trace_bench", "run",
        "--config", str(config_path),
        "--runs-dir", f"{RUNS_DIR}/{MODEL_1_TAG}",
    ],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
    timeout=3600,
)

# Show last 3000 chars of output
output = result.stdout
print(output[-3000:] if len(output) > 3000 else output)
if result.returncode != 0:
    print(f"\nSTDERR (last 2000 chars):\n{result.stderr[-2000:]}")
print(f"\nReturn code: {result.returncode}")

In [ ]:
# Run Model 2 (second model in MODELS list, if available)
if len(MODELS) < 2:
    print("Only one model configured — skipping Model 2 run.")
    MODEL_2 = None
    MODEL_2_TAG = None
else:
    MODEL_2 = MODELS[1]
    MODEL_2_TAG = MODEL_2.split("/")[-1]  # e.g. "deepseek-v3.2" or "deepseek-chat"
    print(f"=== Running BBEH with {MODEL_2} (provider: {PROVIDER}) ===")

    env = dict(os.environ)
    env["PYTHONPATH"] = f"{OPENTRACE_ROOT}:{env.get('PYTHONPATH', '')}"
    env["TRACE_LITELLM_MODEL"] = MODEL_2

    result = subprocess.run(
        [
            sys.executable, "-m", "trace_bench", "run",
            "--config", str(config_path),
            "--runs-dir", f"{RUNS_DIR}/{MODEL_2_TAG}",
        ],
        cwd=str(REPO_ROOT),
        env=env,
        capture_output=True,
        text=True,
        timeout=3600,
    )

    output = result.stdout
    print(output[-3000:] if len(output) > 3000 else output)
    if result.returncode != 0:
        print(f"\nSTDERR (last 2000 chars):\n{result.stderr[-2000:]}")
    print(f"\nReturn code: {result.returncode}")

## 3. Results Table

Load `results.csv` from both model runs and display a unified comparison.

In [ ]:
import pathlib, json, pandas as pd, numpy as np


def latest_run(root):
    """Find the most recent run directory under root."""
    root = pathlib.Path(root)
    if not root.exists():
        return None
    candidates = sorted(
        [p for p in root.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
    )
    for p in reversed(candidates):
        if (p / "results.csv").exists():
            return p
    return None


def load_run(root, model_tag):
    """Load results.csv from a model run, adding model column."""
    run_dir = latest_run(root)
    if run_dir is None:
        print(f"WARNING: No run found for {model_tag} in {root}")
        return None, None
    df = pd.read_csv(run_dir / "results.csv")
    df["model"] = model_tag
    df["objective_mode"] = df["eval_kwargs"].apply(
        lambda x: json.loads(x).get("objective_mode", "?") if isinstance(x, str) else "?"
    )
    return df, run_dir


# Load both runs
frames = []
run_dirs = {}

for model_name in MODELS:
    tag = model_name.split("/")[-1]
    df_model, rd = load_run(f"{RUNS_DIR}/{tag}", tag)
    if df_model is not None:
        frames.append(df_model)
        run_dirs[tag] = rd
        print(f"{tag}: {len(df_model)} jobs, status={df_model['status'].value_counts().to_dict()}")

if not frames:
    print("No results found. Check that the run cells completed.")
    df_all = pd.DataFrame()
else:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal jobs: {len(df_all)}")

if not df_all.empty:
    display_cols = [
        "model", "trainer_id", "objective_mode", "status",
        "score_initial", "score_final", "score_best", "time_seconds",
    ]
    df_all[display_cols]

## 4. Per-Metric Evaluation

Parse TensorBoard event files for `accuracy` and `execution_time_s` progression.
Falls back to re-evaluating the initial code if TF is not installed.

In [ ]:
import struct


def read_tb_scalars(logdir):
    """Read scalar summaries from TensorBoard event files."""
    scalars = []
    logdir = pathlib.Path(logdir)
    if not logdir.exists():
        return scalars
    for event_file in sorted(logdir.glob("events.out.tfevents.*")):
        try:
            data = event_file.read_bytes()
            offset = 0
            while offset < len(data):
                if offset + 12 > len(data):
                    break
                length = struct.unpack("Q", data[offset:offset + 8])[0]
                offset += 12
                if offset + length + 4 > len(data):
                    break
                record = data[offset:offset + length]
                offset += length + 4
                try:
                    from tensorflow.core.util import event_pb2
                    event = event_pb2.Event()
                    event.ParseFromString(record)
                    if event.HasField("summary"):
                        for v in event.summary.value:
                            scalars.append((v.tag, event.step, v.simple_value))
                except ImportError:
                    pass
        except Exception:
            continue
    return scalars


def collect_tb_metrics(run_dir, df_sub):
    """Extract per-metric histories from TB logs for a subset of jobs."""
    records = []
    for _, row in df_sub.iterrows():
        job_id = row["job_id"]
        tb_rel = row.get("tb_logdir", f"jobs/{job_id}/tb")
        tb_dir = pathlib.Path(run_dir) / tb_rel
        scalars = read_tb_scalars(tb_dir)

        acc = [(s, v) for tag, s, v in scalars if "accuracy" in tag.lower()]
        etime = [(s, v) for tag, s, v in scalars if "execution_time" in tag.lower()]

        records.append({
            "job_id": job_id,
            "trainer_id": row["trainer_id"],
            "objective_mode": row["objective_mode"],
            "model": row["model"],
            "accuracy_history": acc,
            "exec_time_history": etime,
            "accuracy_final": acc[-1][1] if acc else None,
            "exec_time_final": etime[-1][1] if etime else None,
        })
    return records


# Collect TB metrics for all runs
all_tb_metrics = []
for tag, rd in run_dirs.items():
    df_sub = df_all[df_all["model"] == tag]
    metrics = collect_tb_metrics(rd, df_sub)
    all_tb_metrics.extend(metrics)

has_tb = any(m["accuracy_final"] is not None for m in all_tb_metrics)
print(f"TensorBoard metrics available: {has_tb}")

In [ ]:
# Re-evaluate initial code to get per-metric baseline scores
sys.path.insert(0, str(OPENTRACE_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from trace_bench.registry import load_task_bundle

metric_rows = []

if not df_all.empty:
    # Get initial score_dict (same for all runs since initial code is fixed)
    bundle = load_task_bundle(
        "internal:multiobjective_bbeh",
        "benchmarks/LLM4AD/benchmark_tasks",
        eval_kwargs={"objective_mode": "weighted"},
    )
    guide = bundle["guide"]
    code = str(bundle["param"].data)
    inputs = bundle["train_dataset"]["inputs"]
    infos = bundle["train_dataset"]["infos"]

    # Evaluate on first example for baseline
    sd_init = guide.get_score_dict(query=inputs[0], response=code, reference=infos[0])
    print(f"Initial code score_dict (example 0): {sd_init}")

    for _, row in df_all.iterrows():
        metric_rows.append({
            "model": row["model"],
            "trainer": row["trainer_id"],
            "mode": row["objective_mode"],
            "accuracy_initial": sd_init["accuracy"],
            "exec_time_initial": sd_init["execution_time_s"],
            "score_initial": row["score_initial"],
            "score_best": row["score_best"],
            "time_s": row["time_seconds"],
        })
    del bundle

df_metrics = pd.DataFrame(metric_rows) if metric_rows else pd.DataFrame()
if not df_metrics.empty:
    print("\nPer-run metrics:")
    df_metrics

## 5. Score Progression

Plot the scalar validation score over training steps, grouped by model.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11})

colors = {
    "BasicSearchAlgorithm": "#2196F3",
    "BeamsearchAlgorithm": "#FF9800",
    "PrioritySearch": "#4CAF50",
}

if df_all.empty:
    print("No data to plot.")
elif has_tb:
    # TB-based progression: facet by model, subplot by mode
    model_tags = sorted(run_dirs.keys())
    n_models = len(model_tags)
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models), squeeze=False)

    for row_idx, model_tag in enumerate(model_tags):
        for col_idx, mode_name in enumerate(["weighted", "pareto"]):
            ax = axes[row_idx, col_idx]
            for m in all_tb_metrics:
                if m["model"] != model_tag or m["objective_mode"] != mode_name:
                    continue
                # Use validation score (scalar)
                rd = run_dirs[model_tag]
                tb_rel = f"jobs/{m['job_id']}/tb"
                scalars = read_tb_scalars(pathlib.Path(rd) / tb_rel)
                val = [(s, v) for tag, s, v in scalars
                       if "validation" in tag.lower() and "score/" not in tag.lower()]
                if not val:
                    val = [(s, v) for tag, s, v in scalars if "objective" in tag.lower()]
                if val:
                    steps, scores = zip(*val)
                    ax.plot(steps, scores, marker="o", ms=4,
                            label=m["trainer_id"],
                            color=colors.get(m["trainer_id"], "gray"))

            ax.set_title(f"{model_tag} — {mode_name}")
            ax.set_xlabel("Training Step")
            ax.set_ylabel("Validation Score")
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

    fig.suptitle("BBEH Boolean Expressions — Score Progression", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    # Fallback: grouped bar chart of score_best
    print("No TensorBoard data — showing bar chart of best scores.")
    model_tags = sorted(df_all["model"].unique())
    fig, axes = plt.subplots(1, len(model_tags), figsize=(7 * len(model_tags), 5),
                             squeeze=False, sharey=True)

    for col_idx, model_tag in enumerate(model_tags):
        ax = axes[0, col_idx]
        sub = df_all[df_all["model"] == model_tag]
        trainers = sub["trainer_id"].unique()
        modes = ["weighted", "pareto"]
        x = np.arange(len(trainers))
        width = 0.35

        for i, mode_name in enumerate(modes):
            mode_sub = sub[sub["objective_mode"] == mode_name]
            vals = [
                mode_sub[mode_sub["trainer_id"] == t]["score_best"].values[0]
                if len(mode_sub[mode_sub["trainer_id"] == t]) > 0 else 0
                for t in trainers
            ]
            ax.bar(x + i * width, vals, width, label=mode_name)

        ax.set_title(model_tag)
        ax.set_xlabel("Trainer")
        ax.set_ylabel("Score (best)")
        ax.set_xticks(x + width / 2)
        short_names = [t.replace("Algorithm", "") for t in trainers]
        ax.set_xticklabels(short_names, fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("BBEH — Best Score by Trainer & Mode", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Per-Metric Progression (accuracy + execution_time_s)

Plot accuracy and execution_time_s trajectories from TensorBoard logs,
or show bar charts if TB is unavailable.

In [ ]:
if df_all.empty:
    print("No data to plot.")
elif has_tb:
    model_tags = sorted(run_dirs.keys())
    n_models = len(model_tags)
    fig, axes = plt.subplots(2 * n_models, 2, figsize=(14, 5 * n_models),
                             squeeze=False)

    for m_idx, model_tag in enumerate(model_tags):
        for col, mode_name in enumerate(["weighted", "pareto"]):
            ax_acc = axes[m_idx * 2, col]
            ax_et = axes[m_idx * 2 + 1, col]

            for m in all_tb_metrics:
                if m["model"] != model_tag or m["objective_mode"] != mode_name:
                    continue
                c = colors.get(m["trainer_id"], "gray")

                if m["accuracy_history"]:
                    steps, vals = zip(*m["accuracy_history"])
                    ax_acc.plot(steps, vals, marker="o", ms=3, label=m["trainer_id"], color=c)

                if m["exec_time_history"]:
                    steps, vals = zip(*m["exec_time_history"])
                    ax_et.plot(steps, vals, marker="s", ms=3, label=m["trainer_id"], color=c)

            ax_acc.set_title(f"{model_tag} — accuracy ({mode_name})")
            ax_acc.set_ylabel("Accuracy (higher = better)")
            ax_et.set_title(f"{model_tag} — execution_time_s ({mode_name})")
            ax_et.set_ylabel("CPU time (lower = better)")
            ax_et.set_xlabel("Step")
            for ax in [ax_acc, ax_et]:
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)

    fig.suptitle("BBEH — Per-Metric Progression", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    # Fallback: bar chart of initial metrics
    if not df_metrics.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for ax, metric, label in [
            (axes[0], "accuracy_initial", "accuracy (initial)"),
            (axes[1], "exec_time_initial", "execution_time_s (initial)"),
        ]:
            for model_tag in df_metrics["model"].unique():
                sub = df_metrics[df_metrics["model"] == model_tag]
                trainers = sub["trainer"].unique()
                vals = [sub[sub["trainer"] == t][metric].values[0]
                        if len(sub[sub["trainer"] == t]) > 0 else 0
                        for t in trainers]
                short = [t.replace("Algorithm", "") for t in trainers]
                ax.bar(short, vals, alpha=0.7, label=model_tag)

            ax.set_title(label)
            ax.legend()
            ax.grid(True, alpha=0.3, axis="y")

        fig.suptitle("BBEH — Initial Metric Values", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("No metric data available.")

## 7. Scatter Plot: accuracy vs execution_time_s

Plot accuracy (x) vs execution_time_s (y) for each run.
Color by trainer, marker by mode, faceted by model.
Ideal: top-left (high accuracy, low time).

In [ ]:
if df_all.empty:
    print("No data to plot.")
else:
    model_tags = sorted(df_all["model"].unique())
    n_models = len(model_tags)
    fig, axes = plt.subplots(1, max(n_models, 1), figsize=(7 * max(n_models, 1), 6),
                             squeeze=False)

    markers = {"weighted": "o", "pareto": "s"}

    for col_idx, model_tag in enumerate(model_tags):
        ax = axes[0, col_idx]
        seen_labels = set()

        # Get scatter data from TB metrics or fallback
        model_metrics = [m for m in all_tb_metrics if m["model"] == model_tag]

        for m in model_metrics:
            acc = m["accuracy_final"]
            et = m["exec_time_final"]

            # Fallback to initial if TB not available
            if acc is None:
                acc = df_metrics.loc[
                    (df_metrics["model"] == model_tag) &
                    (df_metrics["trainer"] == m["trainer_id"]) &
                    (df_metrics["mode"] == m["objective_mode"]),
                    "accuracy_initial"
                ].values
                acc = acc[0] if len(acc) > 0 else None
            if et is None:
                et = df_metrics.loc[
                    (df_metrics["model"] == model_tag) &
                    (df_metrics["trainer"] == m["trainer_id"]) &
                    (df_metrics["mode"] == m["objective_mode"]),
                    "exec_time_initial"
                ].values
                et = et[0] if len(et) > 0 else None

            if acc is None or et is None:
                continue

            trainer_short = m["trainer_id"].replace("Algorithm", "")
            full_label = f"{trainer_short} ({m['objective_mode']})"

            ax.scatter(
                acc, et,
                c=colors.get(m["trainer_id"], "gray"),
                marker=markers.get(m["objective_mode"], "o"),
                s=120, edgecolors="black", linewidths=0.5,
                label=full_label if full_label not in seen_labels else None,
                zorder=5,
            )
            seen_labels.add(full_label)

        ax.set_xlabel("accuracy (higher = better)")
        ax.set_ylabel("execution_time_s (lower = better)")
        ax.set_title(model_tag)
        ax.legend(fontsize=8, loc="best")
        ax.grid(True, alpha=0.3)

    fig.suptitle("BBEH — accuracy vs execution_time_s (Final)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 8. Model Comparison

Side-by-side bar chart comparing final `score_best` per model, grouped by trainer.

In [ ]:
if df_all.empty:
    print("No data to compare.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    model_tags = sorted(df_all["model"].unique())

    for ax, mode_name in zip(axes, ["weighted", "pareto"]):
        mode_df = df_all[df_all["objective_mode"] == mode_name]
        trainers = sorted(mode_df["trainer_id"].unique())
        x = np.arange(len(trainers))
        width = 0.35

        for i, model_tag in enumerate(model_tags):
            sub = mode_df[mode_df["model"] == model_tag]
            vals = [
                sub[sub["trainer_id"] == t]["score_best"].values[0]
                if len(sub[sub["trainer_id"] == t]) > 0 else 0
                for t in trainers
            ]
            ax.bar(x + i * width, vals, width, label=model_tag)

        ax.set_title(f"Mode: {mode_name}")
        ax.set_xlabel("Trainer")
        ax.set_ylabel("Score (best, higher = better)")
        ax.set_xticks(x + width / 2)
        short_names = [t.replace("Algorithm", "") for t in trainers]
        ax.set_xticklabels(short_names, fontsize=9)
        ax.legend()
        ax.grid(True, alpha=0.3, axis="y")

    fig.suptitle("BBEH — Model Comparison (score_best)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Numeric summary
    print("\nScore comparison (score_best):")
    pivot = df_all.pivot_table(
        index=["trainer_id", "objective_mode"],
        columns="model",
        values="score_best",
        aggfunc="first",
    )
    pivot

## 9. Summary

In [ ]:
if df_all.empty:
    print("No results to summarize.")
else:
    print("=== BBEH Multi-Objective Benchmark Summary ===")
    print(f"Total jobs: {len(df_all)}")
    print(f"Status: {df_all['status'].value_counts().to_dict()}")
    print(f"Models: {sorted(df_all['model'].unique())}")
    print(f"Modes: {sorted(df_all['objective_mode'].unique())}")
    print(f"Trainers: {sorted(df_all['trainer_id'].unique())}")

    ok_count = int((df_all["status"] == "ok").sum())
    total = len(df_all)
    print(f"\nSuccess rate: {ok_count}/{total} = {ok_count / total * 100:.0f}%")

    # Best per model x mode
    for model_tag in sorted(df_all["model"].unique()):
        for mode_name in ["weighted", "pareto"]:
            sub = df_all[(df_all["model"] == model_tag) & (df_all["objective_mode"] == mode_name)]
            ok_sub = sub[sub["status"] == "ok"]
            if ok_sub.empty:
                print(f"\n{model_tag} / {mode_name}: no OK jobs")
                continue
            best = ok_sub.loc[ok_sub["score_best"].idxmax()]
            print(f"\n{model_tag} / {mode_name}:")
            print(f"  Best trainer: {best['trainer_id']}")
            print(f"  score_best = {best['score_best']:.4f}")
            print(f"  time = {best['time_seconds']:.1f}s")

---

**Notes**:
- The BBEH task uses the **PAL (Program-Aided Language)** strategy: the trainable
  parameter is Python code, not a prompt. The optimizer proposes code improvements.
- Execution timing uses `time.process_time()` (CPU time), not wall-clock, so
  network latency does not affect the `execution_time_s` objective.
- **API keys**: The setup cell auto-detects the provider from environment variables
  (loaded from `.env` via `python-dotenv` if available):
  - `OPENROUTER_API_KEY` — routes through OpenRouter (Xavier's default).
  - `XAI_API_KEY` + `DEEPSEEK_API_KEY` — direct provider access via LiteLLM.
  - No keys — falls back to stub mode with DummyLLM.
- See also: `06_multiobjective_convex.ipynb` (no LLM needed) and
  `08_multiobjective_gsm8k.ipynb` (LLM agent with token tracking).